# ONS dataset exploration

This notebook downloads the ONS datasets relevant to the housing intelligence project, checks the available files, and profiles each dataset for schema, nulls, and categorical uniqueness.


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
import json
import time
from pathlib import Path
from typing import Any, Optional

import pandas as pd
import requests

# ONS API key (falls back to generic API_KEY)
ONS_API_KEY = os.getenv("ONS_API_KEY") or os.getenv("API_KEY")

ONS_BASE = "https://api.beta.ons.gov.uk/v1"
DATASET_IDS = [
    "house-prices-local-authority",
    "index-private-housing-rental-prices",
    "mid-year-pop-est",
    "ageing-population-estimates",
    "projections-older-people-in-single-households",
    "older-people-net-internal-migration",
    "ashe-tables-7-and-8",
    "labour-market",
    "wellbeing-local-authority",
    "life-expectancy-by-local-authority",
    "gdp-by-local-authority",
    "regional-gdp-by-year",
    "regional-gdp-by-quarter",
    "output-in-the-construction-industry",
    "wellbeing-quarterly",
    "suicides-in-the-uk",
    "ageing-population-projections",
    "older-people-economic-activity",
]

NOTEBOOK_DIR = Path.cwd()
LOCAL_DATA_DIR = NOTEBOOK_DIR / "datasets"
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROFILE_DIR = NOTEBOOK_DIR / "profiles"
PROFILE_DIR.mkdir(parents=True, exist_ok=True)
FULL_SUMMARY_DIR = NOTEBOOK_DIR / "full_summaries"
FULL_SUMMARY_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
def get_json(url: str, params: Optional[dict[str, Any]] = None, headers: Optional[dict[str,str]] = None) -> Any:
    time.sleep(0.5)
    # prefer provided headers, fall back to ONS_API_KEY if available
    if headers is None:
        headers = {}
    if ONS_API_KEY:
        headers.setdefault("Authorization", ONS_API_KEY)
    response = requests.get(url, params=params, headers=headers, timeout=60)
    response.raise_for_status()
    return response.json()


def get_latest_version_url(dataset_id: str) -> str:
    dataset_meta = get_json(f"{ONS_BASE}/datasets/{dataset_id}")
    latest = dataset_meta.get("links", {}).get("latest_version")
    if not latest:
        raise ValueError(f"No latest_version found for dataset: {dataset_id}")
    return latest["href"]


def get_download_url(version_url: str) -> tuple[str, str]:
    version_data = get_json(version_url)
    downloads = version_data.get("downloads", {})

    preferred_order = ["csv", "xlsx", "xls", "json"]
    for key in preferred_order:
        if key in downloads:
            entry = downloads[key]
            href = None
            if isinstance(entry, dict):
                href = entry.get("href") or entry.get("url")
            elif isinstance(entry, str):
                href = entry
            if href:
                return href, key.upper()

    for value in downloads.values():
        if isinstance(value, dict):
            href = value.get("href") or value.get("url")
            if href:
                return href, "FILE"

    raise ValueError(f"No downloadable file found for version: {version_url}")


def save_downloaded_dataset(dataset_id: str, download_url: str, file_type: str) -> Path:
    dataset_dir = LOCAL_DATA_DIR / dataset_id
    dataset_dir.mkdir(parents=True, exist_ok=True)

    suffix_map = {
        "CSV": ".csv",
        "XLSX": ".xlsx",
        "XLS": ".xls",
        "JSON": ".json",
    }
    suffix = suffix_map.get(file_type, ".data")
    local_path = dataset_dir / f"raw{suffix}"

    time.sleep(0.5)
    headers = {}
    if ONS_API_KEY:
        headers["Authorization"] = ONS_API_KEY
    response = requests.get(download_url, headers=headers or None, timeout=120)
    response.raise_for_status()
    local_path.write_bytes(response.content)
    print(f"Saved raw file to: {local_path}")
    return local_path


def load_dataset_from_path(file_path: Path) -> pd.DataFrame:
    suffix = file_path.suffix.lower()

    if suffix == ".csv":
        return pd.read_csv(file_path, low_memory=False)
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(file_path)
    if suffix == ".json":
        with file_path.open("r", encoding="utf-8") as f:
            data = json.load(f)
        if isinstance(data, list):
            return pd.DataFrame(data)
        if isinstance(data, dict):
            if "items" in data and isinstance(data["items"], list):
                return pd.DataFrame(data["items"])
            return pd.json_normalize(data)

    for loader in (pd.read_csv, pd.read_excel):
        try:
            return loader(file_path, low_memory=False) if loader is pd.read_csv else loader(file_path)
        except Exception:
            continue

    try:
        with file_path.open("r", encoding="utf-8") as f:
            data = json.load(f)
        if isinstance(data, list):
            return pd.DataFrame(data)
        if isinstance(data, dict):
            if "items" in data and isinstance(data["items"], list):
                return pd.DataFrame(data["items"])
            return pd.json_normalize(data)
    except Exception:
        pass

    raise ValueError(f"Could not load dataset from file: {file_path}")


def build_full_column_summary(dataset_id: str, df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for column in df.columns:
        s = df[column]
        rows.append(
            {
                "dataset_id": dataset_id,
                "column": column,
                "dtype": str(s.dtype),
                "row_count": int(len(df)),
                "null_count": int(s.isna().sum()),
                "null_pct": round(float(s.isna().mean() * 100), 2),
                "unique_count": int(s.nunique(dropna=True)),
                "min": s.min() if pd.api.types.is_numeric_dtype(s) else None,
                "max": s.max() if pd.api.types.is_numeric_dtype(s) else None,
                "mean": round(float(s.mean()), 4) if pd.api.types.is_numeric_dtype(s) else None,
                "median": round(float(s.median()), 4) if pd.api.types.is_numeric_dtype(s) else None,
            }
        )
    return pd.DataFrame(rows)


def build_full_categorical_breakdown(dataset_id: str, df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    categorical_cols = df.select_dtypes(include=["object", "category", "string"]).columns.tolist()
    for col in categorical_cols:
        s = df[col].dropna()
        value_counts = s.value_counts(dropna=False)
        for value, count in value_counts.items():
            rows.append(
                {
                    "dataset_id": dataset_id,
                    "column": col,
                    "value": value,
                    "count": int(count),
                    "pct_of_rows": round(float(count / len(df) * 100), 4),
                }
            )
    return pd.DataFrame(rows)


def save_full_summaries(dataset_id: str, df: pd.DataFrame) -> tuple[Path, Path]:
    column_summary = build_full_column_summary(dataset_id, df)
    column_path = FULL_SUMMARY_DIR / f"{dataset_id}_column_summary.csv"
    column_summary.to_csv(column_path, index=False)

    categorical_breakdown = build_full_categorical_breakdown(dataset_id, df)
    categorical_path = FULL_SUMMARY_DIR / f"{dataset_id}_categorical_breakdown.csv"
    categorical_breakdown.to_csv(categorical_path, index=False)

    print(f"Saved full column summary to: {column_path}")
    print(f"Saved full categorical breakdown to: {categorical_path}")
    return column_path, categorical_path


In [6]:
for dataset_id in DATASET_IDS:
    print(f"\nProcessing: {dataset_id}")

    try:
        version_url = get_latest_version_url(dataset_id)
        download_url, file_type = get_download_url(version_url)

        dataset_meta = get_json(f"{ONS_BASE}/datasets/{dataset_id}")
        _ = dataset_meta

        raw_path = save_downloaded_dataset(dataset_id, download_url, file_type)
        df = load_dataset_from_path(raw_path)
        save_full_summaries(dataset_id, df)
        print(f"Complete: {dataset_id} ({len(df)} rows, {len(df.columns)} columns)")

    except Exception as exc:
        print(f"ERROR processing {dataset_id}: {exc}")



Processing: house-prices-local-authority
Saved raw file to: d:\hIntel\Exploration\ONS API\datasets\house-prices-local-authority\raw.csv
Saved full column summary to: d:\hIntel\Exploration\ONS API\full_summaries\house-prices-local-authority_column_summary.csv
Saved full categorical breakdown to: d:\hIntel\Exploration\ONS API\full_summaries\house-prices-local-authority_categorical_breakdown.csv
Complete: house-prices-local-authority (794400 rows, 14 columns)

Processing: index-private-housing-rental-prices
Saved raw file to: d:\hIntel\Exploration\ONS API\datasets\index-private-housing-rental-prices\raw.csv
Saved full column summary to: d:\hIntel\Exploration\ONS API\full_summaries\index-private-housing-rental-prices_column_summary.csv
Saved full categorical breakdown to: d:\hIntel\Exploration\ONS API\full_summaries\index-private-housing-rental-prices_categorical_breakdown.csv
Complete: index-private-housing-rental-prices (6870 rows, 8 columns)

Processing: mid-year-pop-est
Saved raw file